In [1]:
!pip install -q openai-whisper
!pip install -q librosa
!pip install -q soundfile
!pip install -q gradio
!pip install -q nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.5 MB/s eta 0:00:00


In [2]:
import whisper
import librosa
import re
import nltk
import gradio as gr

from collections import Counter

In [3]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
print("Loading Whisper model...")

whisper_model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Loading Whisper model...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 122MiB/s]


Whisper model loaded successfully!


In [5]:
def speech_to_text(audio_file):

    result = whisper_model.transcribe(audio_file)

    transcript = result["text"].strip()

    return transcript

In [6]:
positive_words = [
    "happy",
    "good",
    "great",
    "excellent",
    "positive",
    "love",
    "enjoy",
    "calm",
    "confident",
    "excited",
    "hopeful",
    "relaxed",
    "peaceful",
    "success",
    "thankful"
]

negative_words = [
    "sad",
    "bad",
    "angry",
    "stress",
    "stressed",
    "worried",
    "anxious",
    "fear",
    "afraid",
    "tired",
    "lonely",
    "upset",
    "problem",
    "difficult",
    "hopeless"
]


def analyze_sentiment(text):

    words = re.findall(
        r'\b\w+\b',
        text.lower()
    )

    positive_count = sum(
        1 for word in words
        if word in positive_words
    )

    negative_count = sum(
        1 for word in words
        if word in negative_words
    )

    if positive_count > negative_count:
        sentiment = "Positive 😊"

    elif negative_count > positive_count:
        sentiment = "Negative 😟"

    else:
        sentiment = "Neutral 😐"

    return {
        "sentiment": sentiment,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "word_count": len(words)
    }

In [7]:
def generate_feedback(sentiment):

    if "Positive" in sentiment:

        return (
            "Your text shows a generally positive tone. "
            "Continue maintaining positive habits and activities."
        )

    elif "Negative" in sentiment:

        return (
            "Your text contains some negative emotional indicators. "
            "Consider taking a break, talking to someone you trust, "
            "and using healthy coping activities."
        )

    else:

        return (
            "Your text appears generally neutral. "
            "Continue monitoring your feelings and maintaining "
            "healthy daily habits."
        )

In [8]:
def wellness_analyzer(audio_file, text_input):

    # If speech is provided
    if audio_file is not None:

        transcript = speech_to_text(audio_file)

    # Otherwise use typed text
    elif text_input and text_input.strip():

        transcript = text_input.strip()

    else:

        return (
            "Please enter text or record your voice.",
            "",
            "",
            ""
        )

    # Analyze sentiment
    result = analyze_sentiment(transcript)

    # Generate feedback
    feedback = generate_feedback(
        result["sentiment"]
    )

    analysis_report = (
        f"Sentiment: {result['sentiment']}\n"
        f"Word Count: {result['word_count']}\n"
        f"Positive Words: {result['positive_count']}\n"
        f"Negative Words: {result['negative_count']}"
    )

    return (
        transcript,
        analysis_report,
        result["sentiment"],
        feedback
    )

In [9]:
with gr.Blocks(
    title="Mental Wellness Text & Speech Analyzer"
) as app:

    gr.Markdown("""
    # 🧠 Mental Wellness Text & Speech Analyzer

    ### Analyze the general emotional tone of text or speech

    **Text / Speech → Analysis → Sentiment → Feedback**
    """)

    with gr.Row():

        with gr.Column():

            text_input = gr.Textbox(
                label="📝 Enter Your Text",
                placeholder="Type how you are feeling...",
                lines=6
            )

            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎤 Or Record Your Voice"
            )

            analyze_button = gr.Button(
                "🔍 Analyze",
                variant="primary"
            )

        with gr.Column():

            transcript_output = gr.Textbox(
                label="📝 Analyzed Text",
                lines=6
            )

            analysis_output = gr.Textbox(
                label="📊 Analysis Report",
                lines=5
            )

            sentiment_output = gr.Textbox(
                label="😊 Sentiment"
            )

            feedback_output = gr.Textbox(
                label="💡 Supportive Feedback",
                lines=5
            )

    analyze_button.click(
        fn=wellness_analyzer,
        inputs=[
            audio_input,
            text_input
        ],
        outputs=[
            transcript_output,
            analysis_output,
            sentiment_output,
            feedback_output
        ]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8612f1382acc9cf071.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
